In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
        # print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#cell1 %% [code]
import subprocess, sys, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(f"找不到 {pattern}，请检查是否在右侧 Add Data 挂载了对应的依赖库！")

# 100% 照抄原代码：寻找并安装 onnxruntime

ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
if ONNX_WHL.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
    print("ONNX Runtime installed")
else:
    print('没有！')



# 100% 照抄原代码：寻找并降级/升级 TF 2.20 以解决死锁
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)
    print("✅ TF 2.20 installed from wheel")
except Exception as e:
    print(e)

# 导入所有后续需要的包
import onnxruntime as ort
import pandas as pd
import numpy as np
import librosa
from tqdm.auto import tqdm
import re

print("环境配置严格执行完毕！")

ONNX Runtime installed
✅ TF 2.20 installed from wheel
环境配置严格执行完毕！


In [3]:
# %% [code]
# ── Cell 2: 全局需求与配置 ──
from pathlib import Path
import onnxruntime as ort

AUDIO_DIR = Path("/kaggle/input/competitions/birdclef-2026/train_soundscapes")
OUTPUT_DIR = Path("/kaggle/working/extracted_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

onnx_hits = list(Path("/kaggle/input").rglob("perch_v2.onnx"))
if not onnx_hits:
    raise FileNotFoundError("❌ 未找到 perch_v2.onnx")
ONNX_MODEL_PATH = onnx_hits[0]
print(f"✅ 模型路径: {ONNX_MODEL_PATH}")

SR = 32_000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_LEN_SEC = 60
FILE_SAMPLES = SR * FILE_LEN_SEC
N_WINDOWS = FILE_LEN_SEC // WINDOW_SEC

BATCH_FILES = 16
NUM_WORKERS = 4

# ==========================================
# 🚀 内存安全控制：每 1000 个文件清洗一次内存，但全部追加到同一个文件
# ==========================================
CHUNK_SIZE = 1000
# 这次不写 TARGET_GROUPS 了，直接默认跑全量 10000 个！

✅ 模型路径: /kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx


In [4]:
# %% [code]
# ── Cell 3: 扫描文件与初始化推理引擎 ──
import librosa
import soundfile as sf
import numpy as np

# 获取文件
all_files = sorted(list(AUDIO_DIR.glob("**/*.ogg"))) # 递归扫描子文件夹
total_files = len(all_files)
print(f"📁 扫描到总文件数: {total_files}")

chunk_tasks =[]
for group_idx in range(0, total_files, CHUNK_SIZE):
    start_idx = group_idx
    end_idx = min(group_idx + CHUNK_SIZE, total_files)
    chunk_tasks.append({
        "group": group_idx // CHUNK_SIZE,
        "start": start_idx,
        "end": end_idx,
        "files": all_files[start_idx:end_idx]
    })

# 👇 就是改了这里：不再做任何筛选，所有的 chunk 都要跑！
# 注意在你的 cell 5 里，记得把循环改成：for task in chunk_tasks:
print(f"📦 共被划分为 {len(chunk_tasks)} 个批次进行流式处理 (每批最多 {CHUNK_SIZE} 个文件).")

# ================================
# 初始化 ONNX 引擎
# ================================
so = ort.SessionOptions()
so.intra_op_num_threads = 4
so.inter_op_num_threads = 1
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

ort_session = ort.InferenceSession(str(ONNX_MODEL_PATH), sess_options=so, providers=["CPUExecutionProvider"])
input_name = ort_session.get_inputs()[0].name
output_names = [o.name for o in ort_session.get_outputs()]
emb_output_name = next(name for name in output_names if "embedding" in name.lower())

def read_and_tile_audio(path):
    try:
        y, sr = sf.read(str(path), dtype="float32", always_2d=False)
        if y.ndim == 2: y = y.mean(axis=1)
        if sr != SR: y = librosa.resample(y, orig_sr=sr, target_sr=SR)
        if len(y) < FILE_SAMPLES:
            repeats = int(np.ceil(FILE_SAMPLES / len(y)))
            y = np.tile(y, repeats)[:FILE_SAMPLES]
        else:
            y = y[:FILE_SAMPLES]
        return y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    except Exception as e:
        return np.zeros((N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)

📁 扫描到总文件数: 10658
📦 共被划分为 11 个批次进行流式处理 (每批最多 1000 个文件).


In [5]:
# %% [code]
# ── Cell 5: 增量写入单文件，防内存爆炸 ──
import time
import concurrent.futures
import pyarrow as pa
import pyarrow.parquet as pq
import gc

global_t0 = time.time()

# 最终只会生成这唯一的一个文件
final_output_file = OUTPUT_DIR / "train_soundscapes_features.parquet"
parquet_writer = None  # 初始化流式写入器

# 遍历所有被划分好的 Chunk
for task in chunk_tasks:
    group_id = task["group"]
    paths = task["files"]
    n_files = len(paths)
    
    print(f"\n🔥 正在处理组 {group_id} (文件数: {n_files})...")
    
    # 【局部容器】：只装当前这 1000 个文件的数据
    chunk_filenames = []
    chunk_ends =[]
    chunk_embs =[]
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        next_batch_paths = paths[0:BATCH_FILES]
        future_audio =[executor.submit(read_and_tile_audio, p) for p in next_batch_paths]
        
        for i in tqdm(range(0, n_files, BATCH_FILES), desc=f"组 {group_id} 进度"):
            batch_paths = next_batch_paths
            batch_audio = [f.result() for f in future_audio]
            
            next_start = i + BATCH_FILES
            if next_start < n_files:
                next_batch_paths = paths[next_start : next_start + BATCH_FILES]
                future_audio =[executor.submit(read_and_tile_audio, p) for p in next_batch_paths]
            
            x_input = np.vstack(batch_audio).astype(np.float32)
            outputs = ort_session.run([emb_output_name], {input_name: x_input})
            
            for p in batch_paths:
                chunk_filenames.extend([p.name] * N_WINDOWS)
                chunk_ends.extend([t for t in range(5, 65, 5)])
            chunk_embs.append(outputs[0])

    # --- 当前 Chunk 提取完毕，开始流式写入 ---
    print(f"   -> 正在将 组 {group_id} 写入磁盘，并清理内存...")
    final_embs_matrix = np.vstack(chunk_embs).astype(np.float32)
    
    df = pd.DataFrame({
        "filename": chunk_filenames,
        "end_sec": chunk_ends
    })
    df["embedding"] = list(final_embs_matrix)
    
    # 转换为 PyArrow 的 Table
    table = pa.Table.from_pandas(df)
    
    # 如果是第一组，初始化 Writer 并记录数据表结构 (Schema)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(final_output_file, table.schema)
    
    # 增量追加到同一个文件里
    parquet_writer.write_table(table)
    
    # 【极致内存管理】写完立马销毁当前局部变量并强制回收垃圾！
    del chunk_filenames, chunk_ends, chunk_embs, final_embs_matrix, df, table
    gc.collect()

# 所有循环结束，关闭写入器
if parquet_writer is not None:
    parquet_writer.close()

print(f"\n🎉 大功告成! 10000+ 个文件已全量提取至一个文件！")
print(f"⏱️ 总耗时: {(time.time() - global_t0)/3600:.2f} 小时")
print(f"📁 最终唯一文件: {final_output_file.name}")


🔥 正在处理组 0 (文件数: 1000)...


组 0 进度:   0%|          | 0/63 [00:00<?, ?it/s]

   -> 正在将 组 0 写入磁盘，并清理内存...

🔥 正在处理组 1 (文件数: 1000)...


组 1 进度:   0%|          | 0/63 [00:00<?, ?it/s]

   -> 正在将 组 1 写入磁盘，并清理内存...

🔥 正在处理组 2 (文件数: 1000)...


...

   -> 正在将 组 8 写入磁盘，并清理内存...

🔥 正在处理组 9 (文件数: 1000)...


组 9 进度:   0%|          | 0/63 [00:00<?, ?it/s]

   -> 正在将 组 9 写入磁盘，并清理内存...

🔥 正在处理组 10 (文件数: 658)...


组 10 进度:   0%|          | 0/42 [00:00<?, ?it/s]

   -> 正在将 组 10 写入磁盘，并清理内存...

🎉 大功告成! 10000+ 个文件已全量提取至一个文件！
⏱️ 总耗时: 6.30 小时
📁 最终唯一文件: train_soundscapes_features.parquet


In [6]:
# import pandas as pd

# # 你的文件路径
# file_path = "/kaggle/input/notebooks/waozhang/bc2026-perch-featrue1/extracted_features/train_audio_all_features.parquet"

# # 读取全部（小数据量没事），然后只取前5行
# df = pd.read_parquet(file_path)

# # 打印列名
# print("===== 列名 =====")
# print(df.columns.tolist())

# # 打印前5行
# print("\n===== 前5行数据 =====")
# print(df.head())